In [ ]:
import json
import torch
import spacy
from glirel import GLiREL
from gliner import GLiNER
from argparse import Namespace
from huggingface_hub import hf_hub_download

In [ ]:
def load_glirel(model_id, device):
    print(f"Loading GLiREL on {device}...")
    model_file  = hf_hub_download(repo_id=model_id, filename="pytorch_model.bin")
    config_file = hf_hub_download(repo_id=model_id, filename="glirel_config.json")
    with open(config_file) as f:
        cfg = Namespace(**json.load(f))
    if not hasattr(cfg, "span_mode"):
        cfg.span_mode = getattr(cfg, "rel_mode", "marker")
    model = GLiREL(cfg)
    sd = torch.load(model_file, map_location=device)
    w_ck = sd["token_rep_layer.bert_layer.model.embeddings.word_embeddings.weight"]
    w_mo = model.token_rep_layer.bert_layer.model.embeddings.word_embeddings.weight
    if w_ck.shape != w_mo.shape:
        model.token_rep_layer.bert_layer.model.resize_token_embeddings(w_ck.shape[0])
    model.load_state_dict(sd, strict=True)
    return model.to(device).float().eval()

In [ ]:
nlp = spacy.load('en_core_web_sm')

device = 'cuda:1'

#ner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1").to(device) 
ner_model = GLiNER.from_pretrained("knowledgator/gliner-multitask-large-v0.5").to(device) 
rel_model = load_glirel("jackboyla/glirel-large-v0", device)

In [ ]:
def lemmatize_word(word):
    doc = nlp(word)
    lemmas = []
    for token in doc:
        if token.text.lower() == "not" or token.lemma_ == "-PRON-":
            lemmas.append(token.text.lower())
        else:
            lemmas.append(token.lemma_.lower())
            
    return "_".join(lemmas)

In [ ]:
phrases = [
    "Luc Pommeret is working on a project with his interns, Younes and Ava. They all work at LISN.",
    "Patrick has a green coat that he only wears on Sundays.",
    "Frank seems to have figured it out.",
    "Steve Jobs founded Apple.",
    "He doesn't play baseball.", 
    "Chris doesn't like staying with us.",
    "Two boys are playing outside", 
    "A man is playing an instrument.", 
    "He and his friend played at the park, they played football and basketball.",
    "The cat and the dog are in the kitchen.", 
    "Frank's dog doesn't eat fruits, he is allergic",
    "Simon didn't call me back, he is busy.", 
    "Younes is working on a project. His friend is playing a video game.", 
    "I have never seen anyone like Frank, he must be gifted.",
    "He must be sick.", 
    "He is sick.", 
    "He played football after eating.",
    "Two dogs are eating and playing football",
    "Frank is in New York, but he plays football in Manchester.",
    "John is eating a sandwich while Claire eats a pizza",
    "John is eating a ham pizza and a chicken sandwich.",
    "John is eating meat.",
    "They grew up in San Fransisco and now they both live in New York. One of them has an appartment in Manhattan.",
    "Šafov is a village and municipality (obec) in Znojmo District in the South Moravian Region of the Czech Republic."
]

text = phrases[1]
doc = nlp(text)
tokens = [t.text for t in doc]
tokens

In [ ]:
labels_ner = [
    "Person",         
    "Animal",         
    
    "Physical Object",         # Une voiture, la porte, un stylo
    "Property",   # De l'eau, du bois, du sang
    
    "Location or Place",       # La maison, Paris, une boîte, l'extérieur
    "Time or Duration",        # Hier, le matin, 5 minutes
    
    "Event, Action or verb",         # La course, un accident, une victoire
    "State or Condition",      # La faim, la pauvreté, la maladie, cassé
    "Abstract Concept",        # La liberté, le tourisme, la loi
    "Quantity or Measure"      # 5 kilos, beaucoup, 100 euros
]

In [ ]:
entities = ner_model.predict_entities(text, labels=labels_ner)

print("Entities GLiNER:")
for e in entities:
    print(e)

In [ ]:
ner = []

for ent in entities:
    span = doc.char_span(ent["start"], ent["end"])

    if span is None:
        continue

    ner.append([
        span.start,        # token start
        span.end - 1,      # token end
        ent["label"],     # label
        ent["text"]       # text
    ])

print("\nNER for GLiREL:")
print(ner)

In [ ]:
with open('data/relations.json', encoding='utf-8') as file:
    labels_rel = list(json.load(file).values())

LABEL_DESCRIPTIONS = {
    "FormOf":        "is an inflected form of",
    "IsA":           "is a subtype of",
    "PartOf":        "is a part of",
    "HasA":          "has",
    "Contains":      "contains",
    "UsedFor":       "is used for",
    "CapableOf":     "is capable of",
    "AtLocation":    "is located at",
    "Causes":        "causes",
    "HasSubevent":   "has subevent",
    "HasPrerequisite": "has prerequisite",
    "HasProperty":   "has property",
    "MotivatedByGoal": "is motivated by goal",
    "CreatedBy":     "was created by",
    "Synonym":       "is a synonym of",
    "Antonym":       "is an antonym of",
    "SymbolOf":      "is a symbol of",
    "SimilarTo":     "is similar to",
    "MadeOf":        "is made of",
    "ReceivesAction": "receives the action",
    "PerformsAction": "performs the action",
    "HasCondition" : "depends on or is restricted by",
    "HasQuantity" : "has a quantity",
    "AtLocation" : "is located at" ,
    "AtTime" : "took place in, during, or relative to the temporal frame, date, or event",
    "HasContext" : "is contextually related to",
}

with torch.no_grad():
    relations = rel_model.predict_relations(
        tokens,
        LABEL_DESCRIPTIONS.values(),
        threshold=0.0,
        ner=ner,
        top_k=1
    )

In [ ]:
print("\nRelations:")
for r in sorted(relations, key=lambda x: x["score"], reverse=True):
    print(f"{r['head_text']} -> {r['label']} -> {r['tail_text']} ({r['score']:.3f})")